# SIH26080 — Kaggle training pipeline

Runs the `regimerain` pipeline on a **Kaggle CPU notebook** in three stages: `data` → `backtest` (in fold chunks) → `final`.
Full guide: `docs/MODEL_SPEC.md` §24. CLI: `TECHNICAL_SPEC.md` §9.

**Before every run**
1. *Settings → Accelerator:* **None** (CPU). *Settings → Internet:* **On**.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` (only if the repo is private), attached to this notebook.
3. *Add input:* the datasets the stage needs (table below).
4. Set `STAGE` (and `FOLDS` / `COMMIT`) in the next cell.
5. **Save Version → Save & Run All (Commit)**. Don't run long stages interactively.
6. When it finishes: *Output* tab → **New Dataset** (or a new version) with the name in the table.

| STAGE | Attach as input | Save output as dataset |
|---|---|---|
| `data` | `ps26080-imd` | `ps26080-table` |
| `backtest` | `ps26080-imd`, `ps26080-table` | `ps26080-folds-a` / `-b` / `-c` (one per chunk) |
| `final` | `ps26080-imd`, `ps26080-table`, all `ps26080-folds-*` | `ps26080-models` |

In [ ]:
# ---- Parameters: edit these per run ----
STAGE = "data"                       # "data" | "backtest" | "final"
FOLDS = "2016,2017,2018"             # test seasons for STAGE == "backtest"
MODE = "fast"                        # "fast" | "full"  (MODEL_SPEC section 19)
REPO = "https://github.com/shreyashsri79/080.git"
BRANCH = "main"
COMMIT = ""                          # set to a fixed commit SHA once backtesting starts (MODEL_SPEC 24.5)
VARIANTS = "raw,A,B,C"

In [ ]:
# ---- Environment check ----
import json, multiprocessing, os, platform, shutil, socket, subprocess, sys, time
from pathlib import Path
import psutil

SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
LOGS = WORK / "logs"; LOGS.mkdir(parents=True, exist_ok=True)

print("python :", platform.python_version())
print("cores  :", multiprocessing.cpu_count())
print("RAM GB :", round(psutil.virtual_memory().total / 1e9, 1))
for p in (WORK, SCRATCH):
    print(f"{p} free GB:", round(shutil.disk_usage(p).free / 1e9, 1))
try:
    socket.create_connection(("storage.googleapis.com", 443), timeout=5).close()
    print("internet: OK")
except OSError:
    raise SystemExit("Internet is OFF: Settings -> Internet -> On (needs a phone-verified account).")
assert STAGE in {"data", "backtest", "final"}, STAGE

In [ ]:
# ---- Dependencies (the Kaggle image already has numpy/pandas/sklearn/lightgbm/xarray/geopandas) ----
!pip -q install imdlib herbie-data cfgrib eccodes gcsfs "zarr>=2.16,<3" rioxarray pyarrow
!pip freeze > /kaggle/working/logs/pip_freeze.txt
import lightgbm, xarray, sklearn
print("lightgbm", lightgbm.__version__, "| xarray", xarray.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# ---- Get the code (never prints the token) ----
SRC = SCRATCH / "src"
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # public repo, or secret not attached
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
shutil.rmtree(SRC, ignore_errors=True)
r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(SRC)], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("git clone failed (check GITHUB_TOKEN secret / repo URL): " + r.stderr.replace(token or "\0", "***"))
if COMMIT:
    subprocess.run(["git", "-C", str(SRC), "checkout", "-q", COMMIT], check=True)
sha = subprocess.run(["git", "-C", str(SRC), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code at commit", sha)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SRC)], check=True)

In [ ]:
# ---- Locate attached datasets by marker, not by fixed path ----
def find_dirs(marker_glob, max_hits=50):
    return sorted({p.parent for p in list(INPUT.glob(f"**/{marker_glob}"))[:max_hits]})

imd_dirs = sorted({p.parent for p in find_dirs("rain/*.grd")})   # imdlib layout: <imd>/rain/<year>.grd
table_dirs = [p for p in INPUT.glob("**/data/table") if p.is_dir()]
static_dirs = [p for p in INPUT.glob("**/data/static") if p.is_dir()]
fold_dirs = sorted({p.parent for p in INPUT.glob("**/cache/fold=*/DONE")})

print("imd    :", imd_dirs[:1])
print("table  :", table_dirs[:1])
print("static :", static_dirs[:1])
print("folds  :", [p.name for p in fold_dirs])

need = {"data": ["imd"], "backtest": ["imd", "table"], "final": ["imd", "table", "folds"]}[STAGE]
found = {"imd": imd_dirs, "table": table_dirs, "folds": fold_dirs}
missing = [n for n in need if not found[n]]
if missing:
    raise SystemExit(f"Missing inputs for STAGE={STAGE}: {missing}. Add them via 'Add input'.")

In [ ]:
# ---- Kaggle config override (MODEL_SPEC 24.2) ----
import yaml

def kaggle_paths():
    table = table_dirs[0] if table_dirs else WORK / "data" / "table"
    static = static_dirs[0] if static_dirs else WORK / "data" / "static"
    if STAGE == "data":                       # data stage writes fresh tables
        table, static = WORK / "data" / "table", WORK / "data" / "static"
    return {
        "raw": str(SCRATCH / "raw"), "imd": str(imd_dirs[0]),
        "static": str(static), "table": str(table),
        "cache": str(WORK / "cache"), "models": str(WORK / "models"),
        "reports": str(WORK / "reports"), "runs": str(WORK / "runs"),
    }

cfg_override = {"num_threads": 4, "mode": MODE, "paths": kaggle_paths()}
KAGGLE_CFG = SCRATCH / "kaggle.yaml"
KAGGLE_CFG.write_text(yaml.safe_dump(cfg_override, sort_keys=False))
CFG_ARGS = ["--config", str(SRC / "config" / "default.yaml"), "--config", str(KAGGLE_CFG)]
print(KAGGLE_CFG.read_text())

json.dump({"stage": STAGE, "folds": FOLDS, "mode": MODE, "commit": sha, "branch": BRANCH,
           "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "config": cfg_override},
          open(LOGS / "run_info.json", "w"), indent=2)

In [ ]:
# ---- Command runner: streams output, logs time and peak RAM ----
def run(*args):
    cmd = [sys.executable, "-m", "regimerain.cli", *args, *CFG_ARGS]
    print("\n$", " ".join(cmd[2:]), flush=True)
    t0, peak = time.time(), 0
    log = open(LOGS / f"{STAGE}.log", "a")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=SRC)
    proc = psutil.Process(p.pid)
    for line in p.stdout:
        print(line, end=""); log.write(line)
        try:
            peak = max(peak, proc.memory_info().rss + sum(c.memory_info().rss for c in proc.children(recursive=True)))
        except psutil.Error:
            pass
    rc = p.wait()
    dt = time.time() - t0
    summary = f"[{args[0]}] exit={rc} time={dt/60:.1f} min peak_rss={peak/1e9:.1f} GB\n"
    print(summary); log.write(summary); log.close()
    with open(LOGS / "timings.tsv", "a") as f:
        f.write(f"{STAGE}\t{' '.join(args)}\t{rc}\t{dt:.0f}\t{peak/1e9:.2f}\n")
    if rc != 0:
        raise SystemExit(f"Command failed: {' '.join(args)}")

In [ ]:
# ---- STAGE = data: ingest -> static -> labels -> feature tables ----
if STAGE == "data":
    run("ingest", "--years", "2016-2022", "--truth", "imd")
    run("ingest", "--imd-climatology", "1981-2015")
    run("static")
    run("label")
    run("features", "--leads", "1-5")
    run("selftest", "--quick")          # leakage + schema checks on the fresh tables

In [ ]:
# ---- STAGE = backtest: one chunk of LOMO folds ----
if STAGE == "backtest":
    run("backtest", "--variants", VARIANTS, "--mode", MODE, "--folds", FOLDS)
    for y in FOLDS.split(","):
        assert (WORK / "cache" / f"fold={y.strip()}" / "DONE").exists(), f"fold {y} has no DONE marker"

In [ ]:
# ---- STAGE = final: pool all folds -> report + skill gate -> final fit ----
if STAGE == "final":
    dst = WORK / "cache"; dst.mkdir(parents=True, exist_ok=True)
    for d in fold_dirs:                                  # copy fold caches from the attached datasets
        shutil.copytree(d, dst / d.name, dirs_exist_ok=True)
    have = sorted(p.name for p in dst.glob("fold=*") if (p / "DONE").exists())
    print("folds available:", have)
    assert len(have) == 7, "Need all 7 folds (2016-2022) before the final stage."
    run("report")
    run("fit-final")
    rep = WORK / "reports"
    print("reports:", [p.name for p in rep.rglob("verification_report.json")])

In [ ]:
# ---- Output size check (keep /kaggle/working under ~18 GB) ----
def du(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())

total = du(WORK)
for sub in sorted(p for p in WORK.iterdir() if p.is_dir()):
    print(f"{sub.name:10s} {du(sub)/1e9:6.2f} GB")
print(f"TOTAL      {total/1e9:6.2f} GB")
if total > 18e9:
    raise SystemExit("Output exceeds ~18 GB; prune before Kaggle drops it.")
print("\nDone. Next: Output tab -> New Dataset (see the table at the top).")